## 面试问题

Observation 怎么规范化后回喂模型？

## 回答主线

原始 observation（大 JSON、HTML、错误堆栈）不能原样回喂，要规范化：抽取决策字段、剥离噪声与营销/敏感文本、结构化错误。这同时控制上下文膨胀、避免被无关信息带偏、防止把工具自由文本当指令。本 Notebook 用库存查询工具的原始大 JSON，对比原始回喂与规范化回喂的体积、注入面和决策稳定性。

## 真实案例

库存工具返回 `sku/available/eta` 三个决策字段，外加 `promo_text`（营销文案）、`raw_html`、`server_meta` 等无关字段。规范化只保留三个决策字段。数据为教学 JSON，不代表真实库存服务。

In [1]:
import json  # 引入 json 计算体积与拼装 prompt。

raw_observation = {  # 工具返回的原始大 observation。
    "sku": "A-100",  # 决策需要的商品编号。
    "available": 3,  # 决策需要的可用库存。
    "eta": "2 days",  # 决策需要的到货时间。
    "promo_text": "限时5折!立即下单!",  # 无关的营销文案（潜在注入）。
    "raw_html": "<div>banner</div>" * 3,  # 无关的原始 HTML。
    "server_meta": {"region": "cn-1", "trace": "x" * 40},  # 无关的服务器元数据。
}  # 结束原始 observation 定义。

print("原始字段:", list(raw_observation.keys()))  # 展示原始 observation 字段。
print("原始序列化长度:", len(json.dumps(raw_observation, ensure_ascii=False)))  # 展示原始体积。

原始字段: ['sku', 'available', 'eta', 'promo_text', 'raw_html', 'server_meta']
原始序列化长度: 232


## 基线（Baseline）

反面基线：把整个原始 observation 序列化后拼进决策 prompt。prompt 很长，且把营销文案等注入面一并带入。

In [2]:
def build_prompt(observation):  # 把 observation 拼进决策 prompt。
    return "根据观察决定动作: " + json.dumps(observation, ensure_ascii=False)  # 直接序列化整个观察。

raw_prompt = build_prompt(raw_observation)  # 用原始观察构造 prompt。
print("原始 prompt 长度:", len(raw_prompt))  # 展示原始回喂的 prompt 很长。
print("原始 prompt 含营销文案:", "限时5折" in raw_prompt)  # 展示注入面被带入 prompt。

原始 prompt 长度: 242
原始 prompt 含营销文案: True


## 核心实现：字段白名单规范化

用决策字段白名单抽取需要的字段，剥离一切无关内容，得到紧凑、稳定、无注入面的规范化观察。

In [3]:
DECISION_FIELDS = ["sku", "available", "eta"]  # 声明决策真正需要的字段白名单。

def normalize_observation(observation):  # 规范化：只保留决策字段并结构化。
    normalized = {}  # 收集规范化结果。
    for field in DECISION_FIELDS:  # 只遍历白名单字段。
        normalized[field] = observation.get(field)  # 抽取决策相关字段。
    return normalized  # 返回紧凑的规范化观察。

clean_observation = normalize_observation(raw_observation)  # 规范化原始观察。
clean_prompt = build_prompt(clean_observation)  # 用规范化观察构造 prompt。
print("规范化字段:", list(clean_observation.keys()))  # 展示只保留决策字段。
print("规范化 prompt 长度:", len(clean_prompt))  # 展示 prompt 大幅缩短。

规范化字段: ['sku', 'available', 'eta']
规范化 prompt 长度: 59


In [4]:
print("字段数 原始 vs 规范化:", len(raw_observation), len(clean_observation))  # 对比字段数量。
print("prompt 长度 原始 vs 规范化:", len(raw_prompt), len(clean_prompt))  # 对比 prompt 体积。
print("规范化 prompt 含营销文案:", "限时5折" in clean_prompt)  # 展示注入面被剥离。

字段数 原始 vs 规范化: 6 3
prompt 长度 原始 vs 规范化: 242 59
规范化 prompt 含营销文案: False


## 结果解读

规范化把字段从 6 个降到 3 个、prompt 明显变短，且营销文案被剥离——注入面消失。关键：规范化属于确定性外壳，schema 由工具 contract 定义并版本化，不能让模型自己决定看什么。

## 失败案例与修正

原始回喂会被无关文本带偏：下面的朴素决策器看到 prompt 里含「限时5折」就产出无关的 `apply_promo` 动作；规范化剥离营销文案后，它基于库存字段正确产出 `check_stock`。

In [5]:
def decide_from_prompt(prompt):  # 一个会被无关文本带偏的朴素决策器。
    if "限时5折" in prompt:  # 错误地把营销文案当作决策信号。
        return "apply_promo"  # 被带偏产出无关动作。
    if "available" in prompt:  # 正常应基于库存决策。
        return "check_stock"  # 产出正确动作。
    return "unknown"  # 兜底动作。

raw_action = decide_from_prompt(raw_prompt)  # 原始回喂下的决策。
clean_action = decide_from_prompt(clean_prompt)  # 规范化回喂下的决策。
print("原始回喂动作:", raw_action, "(被营销文案带偏)")  # 展示原始回喂产生无关动作。
print("规范化回喂动作:", clean_action, "(基于库存)")  # 展示规范化后决策正确。

原始回喂动作: apply_promo (被营销文案带偏)
规范化回喂动作: check_stock (基于库存)


In [6]:
assert len(clean_observation) == 3  # 规范化后只保留三个决策字段。
assert "promo_text" not in clean_observation  # 营销文案被剥离。
assert len(clean_prompt) < len(raw_prompt)  # 规范化 prompt 更短。
assert raw_action == "apply_promo"  # 原始回喂被营销文案带偏。
assert clean_action == "check_stock"  # 规范化回喂决策正确。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
